In [ ]:
# Trains and evaluates VideoMamba on the RHAD dataset and exports its predictions.

In [ ]:
from pathlib import Path
import gc
import torch

INPUT_ROOT = Path("/kaggle/input")

print("=" * 70)
print("GPU")
print("=" * 70)

print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPU :", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(
        f"GPU {index} :",
        torch.cuda.get_device_name(index)
    )

print("\n" + "=" * 70)
print("CHECKPOINTS DISPONIBLES")
print("=" * 70)

checkpoint_paths = sorted(INPUT_ROOT.rglob("*.pth"))

if not checkpoint_paths:
    raise FileNotFoundError(
        "Aucun fichier .pth trouvé dans /kaggle/input"
    )

for path in checkpoint_paths:
    print(
        path,
        f"— {path.stat().st_size / 1024**2:.1f} Mo"
    )

print("\n" + "=" * 70)
print("CONTENU DES CHECKPOINTS")
print("=" * 70)

for path in checkpoint_paths:
    print(f"\nFICHIER : {path}")

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        print("Type :", type(checkpoint))

        if isinstance(checkpoint, dict):
            print("Clés principales :", list(checkpoint.keys()))

            for key in [
                "epoch",
                "max_accuracy",
                "best_accuracy",
                "best_acc",
                "acc1",
                "accuracy"
            ]:
                if key in checkpoint:
                    print(f"{key} :", checkpoint[key])

            state_dict = checkpoint.get(
                "model",
                checkpoint.get(
                    "module",
                    checkpoint.get("state_dict")
                )
            )

            if isinstance(state_dict, dict):
                print(
                    "Nombre de tenseurs du modèle :",
                    len(state_dict)
                )

                possible_head_keys = [
                    "head.weight",
                    "module.head.weight",
                    "model.head.weight"
                ]

                for key in possible_head_keys:
                    if key in state_dict:
                        print(
                            f"{key} :",
                            tuple(state_dict[key].shape)
                        )

                possible_bias_keys = [
                    "head.bias",
                    "module.head.bias",
                    "model.head.bias"
                ]

                for key in possible_bias_keys:
                    if key in state_dict:
                        print(
                            f"{key} :",
                            tuple(state_dict[key].shape)
                        )

            print(
                "Optimiseur sauvegardé :",
                "optimizer" in checkpoint
            )

            print(
                "Scaler AMP sauvegardé :",
                "scaler" in checkpoint
                or "loss_scaler" in checkpoint
            )

        del checkpoint
        gc.collect()

    except Exception as error:
        print(
            "ERREUR PENDANT LA LECTURE :",
            repr(error)
        )

print("\n" + "=" * 70)
print("RECHERCHE DU DATASET TRAINING")
print("=" * 70)

training_directories = [
    path
    for path in INPUT_ROOT.rglob("Training")
    if path.is_dir()
]

for directory in training_directories:
    class_directories = sorted([
        path
        for path in directory.iterdir()
        if path.is_dir()
    ])

    video_extensions = {
        ".mp4", ".avi", ".mkv", ".mov", ".webm"
    }

    videos = [
        path
        for path in directory.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in video_extensions
        )
    ]

    print("\nDataset possible :", directory)
    print("Nombre de dossiers/classes :", len(class_directories))
    print("Nombre de vidéos :", len(videos))
    print(
        "Classes :",
        [path.name for path in class_directories]
    )

In [ ]:
import os
import sys
import subprocess

print("Python Kaggle :", sys.version)
print("Répertoire :", os.getcwd())

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
!pip install -q uv

!rm -rf /kaggle/working/vm_env

!uv python install 3.10
!uv venv --python 3.10 /kaggle/working/vm_env

!/kaggle/working/vm_env/bin/python --version

In [ ]:
PY = "/kaggle/working/vm_env/bin/python"

!uv pip install \
    --python {PY} \
    --force-reinstall \
    --no-deps \
    "numpy==1.26.4"

In [ ]:
PY = "/kaggle/working/vm_env/bin/python"

!uv pip install \
    --python {PY} \
    --index-url https://download.pytorch.org/whl/cu118 \
    "torch==2.1.1+cu118" \
    "torchvision==0.16.1+cu118"

In [ ]:
import subprocess

PY = "/kaggle/working/vm_env/bin/python"

subprocess.run(
    [
        PY,
        "-c",
        (
            "import torch; "
            "print('PyTorch :', torch.__version__); "
            "print('CUDA PyTorch :', torch.version.cuda); "
            "print('CUDA disponible :', torch.cuda.is_available()); "
            "print('GPU :', torch.cuda.get_device_name(0) "
            "if torch.cuda.is_available() else 'Aucun')"
        ),
    ],
    check=True,
)

In [ ]:
!/kaggle/working/vm_env/bin/python -c \
"import numpy as np, torch; \
print('NumPy :', np.__version__); \
print('PyTorch :', torch.__version__); \
print('CUDA :', torch.version.cuda); \
print('GPU disponible :', torch.cuda.is_available())"

In [ ]:
PY = "/kaggle/working/vm_env/bin/python"

!uv pip install --python {PY} \
    "numpy==1.26.4" \
    "timm==0.4.12" \
    "decord==0.6.0" \
    "einops==0.7.0" \
    "opencv-python-headless==4.8.1.78" \
    "pandas==2.2.1" \
    "scipy==1.12.0" \
    "scikit-learn==1.4.1.post1" \
    "tensorboardX==2.6.2.2" \
    "Pillow==10.1.0" \
    "PyYAML==6.0.1" \
    "tqdm==4.66.1" \
    "termcolor==2.4.0" \
    "fvcore==0.1.5.post20221221" \
    "packaging==24.0" \
    "setuptools==68.2.2" \
    "wheel==0.42.0"

In [ ]:
import subprocess

PY = "/kaggle/working/vm_env/bin/python"

code = """
import numpy as np
import torch
import timm
import decord
import einops
import pandas
import scipy
import sklearn

print("NumPy :", np.__version__)
print("PyTorch :", torch.__version__)
print("CUDA :", torch.version.cuda)
print("GPU disponible :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))
print("timm :", timm.__version__)
print("decord :", decord.__version__)

assert np.__version__ == "1.26.4"
assert torch.__version__ == "2.1.1+cu118"
assert torch.version.cuda == "11.8"
assert torch.cuda.is_available()
assert timm.__version__ == "0.4.12"

print("DÉPENDANCES DE BASE OK")
"""

subprocess.run(
    [PY, "-c", code],
    check=True,
)

In [ ]:
from pathlib import Path
import hashlib
import urllib.request

WHEELS_DIR = Path("/kaggle/working/videomamba_wheels")
WHEELS_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "causal_conv1d-1.0.0-cp310-cp310-linux_x86_64.whl": {
        "url": (
            "https://huggingface.co/spaces/OpenGVLab/VideoMamba/"
            "resolve/main/causal_conv1d-1.0.0-cp310-cp310-linux_x86_64.whl"
        ),
        "sha256": "78328bff9f0cf4814aa3c4029d63aa75128694e07ddae688b16215e3d8a2e7e7",
    },
    "mamba_ssm-1.0.1-cp310-cp310-linux_x86_64.whl": {
        "url": (
            "https://huggingface.co/spaces/OpenGVLab/VideoMamba/"
            "resolve/main/mamba_ssm-1.0.1-cp310-cp310-linux_x86_64.whl"
        ),
        "sha256": "96c19c64be06f0bb5607c6a2406fdf7289a0d83037ea005ec56a2e4ae5d40ec7",
    },
}


def calculate_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


for filename, information in files.items():
    destination = WHEELS_DIR / filename

    # Retélécharge le fichier s'il est absent ou incomplet.
    if (
        not destination.exists()
        or calculate_sha256(destination) != information["sha256"]
    ):
        destination.unlink(missing_ok=True)
        print(f"Téléchargement : {filename}")
        urllib.request.urlretrieve(information["url"], destination)

    actual_sha256 = calculate_sha256(destination)

    print(f"\nFichier : {filename}")
    print(f"Taille : {destination.stat().st_size / 1024**2:.2f} Mo")
    print(f"SHA256 correct : {actual_sha256 == information['sha256']}")

    assert actual_sha256 == information["sha256"], (
        f"Le téléchargement de {filename} est incorrect."
    )

print("\nROUES VIDEOMAMBA TÉLÉCHARGÉES ET VÉRIFIÉES")

In [ ]:
import glob, subprocess
PY = "/kaggle/working/vm_env/bin/python"

wheels = glob.glob("/kaggle/working/videomamba_wheels/*.whl")
print("Wheels à installer :", wheels)

subprocess.run(
    ["uv", "pip", "install", "--python", PY, "--link-mode", "copy", *wheels],
    check=True,
)
print("Wheels installées.")

In [ ]:
import subprocess

PY = "/kaggle/working/vm_env/bin/python"

# Version compatible avec mamba-ssm 1.0.1
subprocess.run(
    [
        "uv", "pip", "install",
        "--python", PY,
        "--link-mode", "copy",
        "transformers==4.36.2",
        "numpy==1.26.4",
    ],
    check=True,
)

# Test réel de Mamba sur le GPU
test_code = r"""
import inspect
import torch
import transformers

from importlib.metadata import version
from mamba_ssm.modules.mamba_simple import Mamba

print("Transformers :", transformers.__version__)
print("causal-conv1d :", version("causal-conv1d"))
print("mamba-ssm :", version("mamba-ssm"))

signature = inspect.signature(Mamba.__init__)
print("Argument bimamba disponible :", "bimamba" in signature.parameters)

assert transformers.__version__ == "4.36.2"
assert version("causal-conv1d") == "1.0.0"
assert version("mamba-ssm") == "1.0.1"
assert "bimamba" in signature.parameters
assert torch.cuda.is_available()

model = Mamba(
    d_model=64,
    bimamba=True,
).cuda().eval()

x = torch.randn(1, 16, 64, device="cuda")

with torch.no_grad():
    y = model(x)

print("Entrée :", tuple(x.shape))
print("Sortie :", tuple(y.shape))

assert y.shape == x.shape
assert torch.isfinite(y).all()

print("MAMBA GPU VALIDÉ")
"""

subprocess.run([PY, "-c", test_code], check=True)

In [ ]:
from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vm_env/bin/python"

REPO_DIR = Path("/kaggle/working/VideoMamba")
VIDEO_SM_DIR = REPO_DIR / "videomamba" / "video_sm"

# Supprime seulement une éventuelle ancienne copie du dépôt.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Téléchargement du code officiel.
subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/OpenGVLab/VideoMamba.git",
        str(REPO_DIR),
    ],
    check=True,
)

assert VIDEO_SM_DIR.exists(), (
    f"Dossier introuvable : {VIDEO_SM_DIR}"
)

# Test de création du vrai modèle VideoMamba Middle.
test_code = r"""
import torch

from timm.models import create_model
import models

model = create_model(
    "videomamba_middle",
    pretrained=False,
    num_classes=12,
    num_frames=16,
    drop_path_rate=0.1,
)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Modèle :", model.__class__.__name__)
print("Nombre de classes :", model.num_classes)
print("Nombre de paramètres :", f"{number_of_parameters:,}")
print("VIDEOMAMBA MIDDLE CRÉÉ CORRECTEMENT")
"""

subprocess.run(
    [PY, "-c", test_code],
    cwd=str(VIDEO_SM_DIR),
    check=True,
)

In [ ]:
import subprocess

PY = "/kaggle/working/vm_env/bin/python"

download_code = r"""
from pathlib import Path
import torch
from huggingface_hub import hf_hub_download

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = hf_hub_download(
    repo_id="OpenGVLab/VideoMamba",
    filename="videomamba_m16_k400_f16_res224.pth",
    local_dir=str(CHECKPOINT_DIR),
)

checkpoint_path = Path(checkpoint_path)

print("Fichier :", checkpoint_path.name)
print(f"Taille : {checkpoint_path.stat().st_size / 1024**2:.2f} Mo")

assert checkpoint_path.exists()
assert checkpoint_path.stat().st_size > 100 * 1024**2

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
)

print("Type du checkpoint :", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Clés principales :", list(checkpoint.keys())[:10])

print("CHECKPOINT VIDEOMAMBA MIDDLE PRÊT")
"""

subprocess.run(
    [PY, "-c", download_code],
    check=True,
)

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import csv
import random
import re

# -----------------------------
# Configuration
# -----------------------------
BASE = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/training/Training"
)

ANNO_DIR = Path("/kaggle/working/annotations")
ANNO_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mkv", ".mov", ".webm"
}

SEED = 42
CLIPS_PER_SOURCE = 5

assert BASE.exists(), f"Dataset introuvable : {BASE}"

# -----------------------------
# Classes
# -----------------------------
classes = sorted(
    folder.name
    for folder in BASE.iterdir()
    if folder.is_dir()
)

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(classes)
}

print("Nombre de classes :", len(classes))
print("Classes :", class_to_idx)

assert len(classes) == 12, (
    f"12 classes attendues, mais {len(classes)} trouvées."
)

# -----------------------------
# Identification de la vidéo source
# -----------------------------
def extract_group_id(video_path: Path) -> str:
    """
    Exemple :
    1 (11).mp4, 1 (12).mp4, ..., 1 (15).mp4
    sont placés dans le même groupe.

    Les clips sont regroupés par blocs de 5.
    """

    stem = video_path.stem
    relative_parent = video_path.parent.relative_to(BASE)

    match = re.match(r"^(.*?)\s*\((\d+)\)$", stem)

    if match:
        prefix = match.group(1).strip()
        clip_number = int(match.group(2))

        block_number = (
            clip_number - 1
        ) // CLIPS_PER_SOURCE

        return (
            f"{relative_parent}/"
            f"{prefix}/block_{block_number:04d}"
        )

    # Pour les fichiers ne respectant pas ce format,
    # chaque vidéo est considérée comme un groupe indépendant.
    return str(video_path.relative_to(BASE))


# -----------------------------
# Regroupement par classe
# -----------------------------
groups_by_class = defaultdict(
    lambda: defaultdict(list)
)

total_videos = 0

for class_name in classes:
    class_directory = BASE / class_name
    label = class_to_idx[class_name]

    video_files = sorted(
        path
        for path in class_directory.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in VIDEO_EXTENSIONS
        )
    )

    for video_path in video_files:
        group_id = extract_group_id(video_path)

        groups_by_class[class_name][group_id].append(
            (str(video_path), label)
        )

        total_videos += 1

print("Nombre total de vidéos :", total_videos)

assert total_videos > 0, "Aucune vidéo trouvée."

# -----------------------------
# Split par groupes : 80 / 10 / 10
# -----------------------------
rng = random.Random(SEED)

train_samples = []
val_samples = []
test_samples = []

train_groups = set()
val_groups = set()
test_groups = set()

split_manifest = []

for class_name in classes:
    class_groups = list(
        groups_by_class[class_name].items()
    )

    rng.shuffle(class_groups)

    number_of_groups = len(class_groups)

    if number_of_groups < 3:
        raise RuntimeError(
            f"La classe {class_name} ne possède que "
            f"{number_of_groups} groupes. Il en faut au moins 3."
        )

    number_of_test_groups = max(
        1,
        round(number_of_groups * 0.10)
    )

    number_of_val_groups = max(
        1,
        round(number_of_groups * 0.10)
    )

    # Garantit qu'il reste au moins un groupe pour train
    while (
        number_of_test_groups
        + number_of_val_groups
        >= number_of_groups
    ):
        if number_of_test_groups > 1:
            number_of_test_groups -= 1
        elif number_of_val_groups > 1:
            number_of_val_groups -= 1
        else:
            break

    test_part = class_groups[
        :number_of_test_groups
    ]

    val_part = class_groups[
        number_of_test_groups:
        number_of_test_groups
        + number_of_val_groups
    ]

    train_part = class_groups[
        number_of_test_groups
        + number_of_val_groups:
    ]

    for split_name, groups, samples, group_set in [
        (
            "train",
            train_part,
            train_samples,
            train_groups,
        ),
        (
            "val",
            val_part,
            val_samples,
            val_groups,
        ),
        (
            "test",
            test_part,
            test_samples,
            test_groups,
        ),
    ]:
        for group_id, group_samples in groups:
            group_set.add(group_id)
            samples.extend(group_samples)

            split_manifest.append({
                "split": split_name,
                "class_name": class_name,
                "label": class_to_idx[class_name],
                "group_id": group_id,
                "number_of_clips": len(group_samples),
            })

# Mélange l'ordre des vidéos
rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)

# -----------------------------
# Vérification des fuites
# -----------------------------
assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

train_paths = {
    path for path, _ in train_samples
}

val_paths = {
    path for path, _ in val_samples
}

test_paths = {
    path for path, _ in test_samples
}

assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)

assert (
    len(train_samples)
    + len(val_samples)
    + len(test_samples)
    == total_videos
)

# -----------------------------
# Écriture CSV
# Format : chemin,label
# -----------------------------
def write_annotation_file(
    destination: Path,
    samples,
):
    with destination.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.writer(file)

        for video_path, label in samples:
            writer.writerow([
                video_path,
                label,
            ])


write_annotation_file(
    ANNO_DIR / "train.csv",
    train_samples,
)

write_annotation_file(
    ANNO_DIR / "val.csv",
    val_samples,
)

write_annotation_file(
    ANNO_DIR / "test.csv",
    test_samples,
)

# Mapping labels/classes
with (
    ANNO_DIR / "label_mapping.csv"
).open(
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.writer(file)

    writer.writerow([
        "label",
        "class_name",
    ])

    for class_name, label in class_to_idx.items():
        writer.writerow([
            label,
            class_name,
        ])

# Manifeste des groupes
with (
    ANNO_DIR / "split_groups.csv"
).open(
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "split",
            "class_name",
            "label",
            "group_id",
            "number_of_clips",
        ],
    )

    writer.writeheader()
    writer.writerows(split_manifest)

# -----------------------------
# Résultats
# -----------------------------
def class_distribution(samples):
    counts = Counter(
        label
        for _, label in samples
    )

    return {
        classes[label]: counts[label]
        for label in range(len(classes))
    }


print("\n--- RÉSULTAT DU SPLIT ---")
print("Train :", len(train_samples))
print("Validation :", len(val_samples))
print("Test :", len(test_samples))

print("\nGroupes train :", len(train_groups))
print("Groupes validation :", len(val_groups))
print("Groupes test :", len(test_groups))

print("\nFuites train/val :",
      len(train_groups & val_groups))

print("Fuites train/test :",
      len(train_groups & test_groups))

print("Fuites val/test :",
      len(val_groups & test_groups))

print("\nRépartition train :")
for class_name, count in class_distribution(
    train_samples
).items():
    print(f"{class_name:25s} : {count}")

print("\nRépartition validation :")
for class_name, count in class_distribution(
    val_samples
).items():
    print(f"{class_name:25s} : {count}")

print("\nRépartition test :")
for class_name, count in class_distribution(
    test_samples
).items():
    print(f"{class_name:25s} : {count}")

print("\nFichiers créés :")
for filename in [
    "train.csv",
    "val.csv",
    "test.csv",
    "label_mapping.csv",
    "split_groups.csv",
]:
    path = ANNO_DIR / filename

    print(
        filename,
        "->",
        path.exists(),
        f"({path.stat().st_size} octets)",
    )

print("\nANNOTATIONS CRÉÉES SANS FUITE")

In [ ]:
from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vm_env/bin/python"

TRITON_BUILD_FILE = Path(
    "/kaggle/working/vm_env/lib/python3.10/"
    "site-packages/triton/common/build.py"
)

DRIVER_DIR = "/usr/local/nvidia/lib64"
LIBCUDA = Path(DRIVER_DIR) / "libcuda.so"

assert TRITON_BUILD_FILE.exists(), TRITON_BUILD_FILE
assert LIBCUDA.exists(), LIBCUDA

# Sauvegarde du fichier original
backup_path = TRITON_BUILD_FILE.with_suffix(".py.backup")

if not backup_path.exists():
    shutil.copy2(TRITON_BUILD_FILE, backup_path)

text = TRITON_BUILD_FILE.read_text()

marker = "# KAGGLE_LIBCUDA_FIX"

if marker not in text:
    old = """    dirs = [os.path.dirname(loc) for loc in locs]
"""

    new = """    dirs = [os.path.dirname(loc) for loc in locs]

    # KAGGLE_LIBCUDA_FIX
    kaggle_driver_dir = "/usr/local/nvidia/lib64"
    if os.path.exists(os.path.join(kaggle_driver_dir, "libcuda.so")):
        dirs.insert(0, kaggle_driver_dir)
"""

    if old not in text:
        raise RuntimeError(
            "Impossible de trouver la ligne à corriger dans build.py"
        )

    text = text.replace(old, new, 1)
    TRITON_BUILD_FILE.write_text(text)

    print("Patch Triton appliqué.")
else:
    print("Patch Triton déjà présent.")

# Supprime seulement le cache des compilations Triton échouées
shutil.rmtree(
    Path.home() / ".triton" / "cache",
    ignore_errors=True,
)

# Test du vrai RMSNorm utilisé par VideoMamba
test_code = r"""
import torch

from triton.common.build import libcuda_dirs
from mamba_ssm.ops.triton.layernorm import RMSNorm

print("Dossiers détectés par Triton :", libcuda_dirs())

assert "/usr/local/nvidia/lib64" in libcuda_dirs()

layer = RMSNorm(576).cuda().eval()

x = torch.randn(
    64,
    576,
    device="cuda",
    dtype=torch.float32,
)

with torch.no_grad():
    y = layer(x)

print("Entrée :", tuple(x.shape))
print("Sortie :", tuple(y.shape))

assert y.shape == x.shape
assert torch.isfinite(y).all()

print("TRITON RMSNORM VALIDÉ")
"""

subprocess.run(
    [PY, "-c", test_code],
    check=True,
)

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

PY = "/kaggle/working/vm_env/bin/python"

# Trouver l'emplacement réel de libcuda.so.1
ldconfig_command = shutil.which("ldconfig") or "/sbin/ldconfig"

ldconfig_output = subprocess.check_output(
    [ldconfig_command, "-p"],
    text=True,
)

candidates = []

for line in ldconfig_output.splitlines():
    if "libcuda.so.1" in line and "=>" in line:
        candidate = Path(
            line.split("=>", 1)[1].strip()
        )

        if candidate.exists():
            candidates.append(candidate)

# Emplacements de secours possibles sur Kaggle
if not candidates:
    for directory in [
        Path("/usr/lib/x86_64-linux-gnu"),
        Path("/lib/x86_64-linux-gnu"),
        Path("/usr/lib64-nvidia"),
        Path("/usr/local/nvidia/lib64"),
    ]:
        candidate = directory / "libcuda.so.1"

        if candidate.exists():
            candidates.append(candidate)

if not candidates:
    raise FileNotFoundError(
        "Impossible de trouver libcuda.so.1."
    )

libcuda_so1 = candidates[0]
libcuda_link = libcuda_so1.parent / "libcuda.so"

print("libcuda.so.1 trouvé :", libcuda_so1)
print("Fichier réel :", libcuda_so1.resolve())

# Supprime seulement un éventuel lien cassé
if os.path.lexists(libcuda_link):
    if libcuda_link.resolve() == libcuda_so1.resolve():
        print("Le lien libcuda.so existe déjà et il est correct.")
    else:
        libcuda_link.unlink()
        os.symlink(
            str(libcuda_so1.resolve()),
            str(libcuda_link),
        )
        print("Ancien lien remplacé :", libcuda_link)
else:
    os.symlink(
        str(libcuda_so1.resolve()),
        str(libcuda_link),
    )
    print("Lien créé :", libcuda_link)

assert libcuda_link.exists()

# Test réel du RMSNorm Triton utilisé par VideoMamba
test_code = r"""
import torch

from triton.common.build import libcuda_dirs
from mamba_ssm.ops.triton.layernorm import RMSNorm

print("Dossiers libcuda détectés :", libcuda_dirs())

layer = RMSNorm(576).cuda().eval()
x = torch.randn(
    64,
    576,
    device="cuda",
    dtype=torch.float32,
)

with torch.no_grad():
    y = layer(x)

print("Entrée :", tuple(x.shape))
print("Sortie :", tuple(y.shape))

assert y.shape == x.shape
assert torch.isfinite(y).all()

print("TRITON RMSNORM VALIDÉ")
"""

subprocess.run(
    [PY, "-c", test_code],
    check=True,
)

In [ ]:
# CELLULE 16B — ÉVALUATION DU CHECKPOINT BEST

from pathlib import Path
import csv
import os
import shutil
import subprocess
import torch

# ============================================================
# 1. CHEMINS
# ============================================================

PY = "/kaggle/working/vm_env/bin/python"

VIDEO_SM_DIR = Path(
    "/kaggle/working/VideoMamba/videomamba/video_sm"
)

ANNOTATIONS_DIR = Path(
    "/kaggle/working/annotations"
)

BEST_CHECKPOINT = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "bestvideomae/checkpoint-best (8).pth"
)

OUTPUT_DIR = Path(
    "/kaggle/working/videomamba_best_evaluation"
)

# Recréer un dossier de résultats propre
shutil.rmtree(
    OUTPUT_DIR,
    ignore_errors=True,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# 2. VÉRIFICATIONS AVANT LE TEST
# ============================================================

assert Path(PY).exists(), (
    "L'environnement vm_env est absent."
)

assert VIDEO_SM_DIR.exists(), (
    f"Le dépôt VideoMamba est absent : {VIDEO_SM_DIR}"
)

assert BEST_CHECKPOINT.exists(), (
    f"Le checkpoint est absent : {BEST_CHECKPOINT}"
)

for filename in [
    "train.csv",
    "val.csv",
    "test.csv",
]:
    file_path = ANNOTATIONS_DIR / filename

    assert file_path.exists(), (
        f"Fichier d'annotations absent : {file_path}"
    )

# Vérification du nombre de vidéos test
with (
    ANNOTATIONS_DIR / "test.csv"
).open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    test_rows = list(csv.reader(file))

print("Nombre de vidéos test :", len(test_rows))

assert len(test_rows) == 335, (
    f"335 vidéos test attendues, "
    f"mais {len(test_rows)} ont été trouvées."
)

# Vérification du checkpoint
checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(checkpoint, dict)
assert "model" in checkpoint
assert "epoch" in checkpoint

saved_epoch = checkpoint["epoch"]
state_dict = checkpoint["model"]

print("Époque enregistrée :", saved_epoch)
print(
    "Forme de head.weight :",
    tuple(state_dict["head.weight"].shape),
)

assert saved_epoch == 27, (
    f"L'époque attendue était 27, "
    f"mais le checkpoint contient {saved_epoch}."
)

assert tuple(
    state_dict["head.weight"].shape
) == (12, 576), (
    "La tête du modèle ne correspond pas "
    "à VideoMamba Middle avec 12 classes."
)

del checkpoint

print("Checkpoint validé.")
print("Split test validé.")

# ============================================================
# 3. COMMANDE D'ÉVALUATION
# ============================================================

command = [
    PY,

    "-m",
    "torch.distributed.launch",

    # Un seul processus et un seul GPU
    "--nproc_per_node=1",
    "--use_env",
    "--master_port=29511",

    "run_class_finetuning.py",

    # --------------------------
    # Architecture
    # --------------------------
    "--model",
    "videomamba_middle",

    "--nb_classes",
    "12",

    # Charger les poids entraînés
    "--finetune",
    str(BEST_CHECKPOINT),

    # --------------------------
    # Dataset
    # --------------------------
    "--data_path",
    str(ANNOTATIONS_DIR),

    "--prefix",
    "",

    "--data_set",
    "Kinetics_sparse",

    "--split",
    ",",

    # --------------------------
    # Résultats
    # --------------------------
    "--output_dir",
    str(OUTPUT_DIR),

    "--log_dir",
    str(OUTPUT_DIR),

    # --------------------------
    # Paramètres du modèle
    # identiques à l'entraînement
    # --------------------------
    "--batch_size",
    "4",

    "--num_sample",
    "1",

    "--input_size",
    "224",

    "--short_side_size",
    "224",

    "--num_frames",
    "16",

    "--orig_t_size",
    "16",

    "--tubelet_size",
    "1",

    "--sampling_rate",
    "4",

    "--drop_path",
    "0.1",

    # --------------------------
    # Protocole de test
    # --------------------------
    "--test_num_segment",
    "1",

    "--test_num_crop",
    "1",

    "--num_workers",
    "2",

    # Nécessaire au sampler du test
    "--dist_eval",

    # Empêche le script de chercher
    # un autre checkpoint automatiquement
    "--no_auto_resume",

    # Ne sauvegarde aucun nouveau checkpoint
    "--no_save_ckpt",

    # Évaluation seulement
    "--eval",
]

# ============================================================
# 4. ENVIRONNEMENT GPU
# ============================================================

environment = os.environ.copy()

# Utilisation d'un seul des deux GPU T4
environment["CUDA_VISIBLE_DEVICES"] = "0"
environment["OMP_NUM_THREADS"] = "1"

print("\n" + "=" * 70)
print("ÉVALUATION DE VIDEOMAMBA MIDDLE")
print("=" * 70)
print("Checkpoint :", BEST_CHECKPOINT)
print("Époque du checkpoint :", saved_epoch)
print("Vidéos de test :", len(test_rows))
print("GPU utilisé : Tesla T4 numéro 0")
print("Aucun entraînement ne sera lancé.")
print("=" * 70 + "\n")

# ============================================================
# 5. LANCEMENT
# ============================================================

result = subprocess.run(
    command,
    cwd=str(VIDEO_SM_DIR),
    env=environment,
    check=False,
)

if result.returncode != 0:
    raise RuntimeError(
        "L'évaluation a échoué. "
        f"Code de retour : {result.returncode}"
    )

# ============================================================
# 6. AFFICHAGE DES RÉSULTATS
# ============================================================

print("\n" + "=" * 70)
print("ÉVALUATION TERMINÉE")
print("=" * 70)

log_path = OUTPUT_DIR / "log.txt"

if log_path.exists():
    print("\nContenu de log.txt :\n")
    print(
        log_path.read_text(
            encoding="utf-8"
        )
    )
else:
    print("Le fichier log.txt n'a pas été créé.")

print("\nFichiers générés :")

for file_path in sorted(
    OUTPUT_DIR.rglob("*")
):
    if file_path.is_file():
        print(
            file_path.relative_to(OUTPUT_DIR),
            "—",
            f"{file_path.stat().st_size / 1024:.1f} Ko",
        )

In [ ]:
# ANALYSE DES 335 PRÉDICTIONS

from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# ============================================================
# 1. CHEMINS
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/videomamba_best_evaluation"
)

PREDICTIONS_FILE = OUTPUT_DIR / "0.txt"

ANNOTATIONS_DIR = Path(
    "/kaggle/working/annotations"
)

TEST_CSV = ANNOTATIONS_DIR / "test.csv"
LABEL_MAPPING_CSV = ANNOTATIONS_DIR / "label_mapping.csv"

assert PREDICTIONS_FILE.exists(), PREDICTIONS_FILE
assert TEST_CSV.exists(), TEST_CSV
assert LABEL_MAPPING_CSV.exists(), LABEL_MAPPING_CSV

# ============================================================
# 2. LECTURE DU MAPPING DES CLASSES
# ============================================================

mapping_df = pd.read_csv(LABEL_MAPPING_CSV)

mapping_df = mapping_df.sort_values(
    "label"
).reset_index(drop=True)

class_names = mapping_df["class_name"].tolist()
labels = mapping_df["label"].astype(int).tolist()

print("Classes utilisées :")

for label, class_name in zip(labels, class_names):
    print(f"{label:2d} -> {class_name}")

assert len(class_names) == 12

# ============================================================
# 3. LECTURE DE TEST.CSV
# ============================================================

test_rows = []

with TEST_CSV.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    reader = csv.reader(file)

    for row in reader:
        if len(row) != 2:
            raise ValueError(
                f"Ligne test.csv incorrecte : {row}"
            )

        test_rows.append({
            "path": row[0],
            "true_label_csv": int(row[1]),
        })

print("\nNombre de fichiers dans test.csv :", len(test_rows))

assert len(test_rows) == 335

# ============================================================
# 4. LECTURE DE 0.TXT
# ============================================================

lines = PREDICTIONS_FILE.read_text(
    encoding="utf-8"
).splitlines()

# La première ligne contient seulement les métriques internes
prediction_lines = [
    line.strip()
    for line in lines[1:]
    if line.strip()
]

records = []

for line_number, line in enumerate(
    prediction_lines,
    start=2,
):
    try:
        # Sépare la fin :
        # label, numéro du segment et numéro du crop
        left_part, metadata = line.rsplit(
            "] ",
            1,
        )

        metadata_parts = metadata.split()

        if len(metadata_parts) != 3:
            raise ValueError(
                f"Métadonnées incorrectes : {metadata}"
            )

        true_label = int(metadata_parts[0])
        chunk_number = int(metadata_parts[1])
        crop_number = int(metadata_parts[2])

        # Sépare l'identifiant et les logits
        video_id, logits_text = left_part.split(
            " [",
            1,
        )

        logits = np.fromstring(
            logits_text,
            dtype=np.float64,
            sep=",",
        )

        if len(logits) != 12:
            raise ValueError(
                f"{len(logits)} logits trouvés au lieu de 12."
            )

        prediction = int(np.argmax(logits))

        top5_predictions = np.argsort(
            -logits
        )[:5]

        # Softmax pour obtenir une confiance lisible
        shifted_logits = logits - np.max(logits)
        probabilities = np.exp(shifted_logits)
        probabilities /= probabilities.sum()

        confidence = float(
            probabilities[prediction]
        )

        records.append({
            "video_id": video_id.strip(),
            "true_label": true_label,
            "predicted_label": prediction,
            "correct": int(
                prediction == true_label
            ),
            "top5_correct": int(
                true_label in top5_predictions
            ),
            "confidence": confidence,
            "chunk_number": chunk_number,
            "crop_number": crop_number,
            "logits": logits,
        })

    except Exception as error:
        raise RuntimeError(
            f"Erreur à la ligne {line_number} :\n"
            f"{line}\n\n"
            f"{error}"
        ) from error

print("Nombre de prédictions lues :", len(records))

assert len(records) == 335, (
    f"335 prédictions attendues, "
    f"mais {len(records)} trouvées."
)

# ============================================================
# 5. VÉRIFICATION DE L'ORDRE
# ============================================================

id_matches = 0
label_matches = 0

for record, test_row in zip(records, test_rows):
    expected_id = (
        Path(test_row["path"])
        .name
        .split(".")[0]
    )

    if record["video_id"] == expected_id:
        id_matches += 1

    if (
        record["true_label"]
        == test_row["true_label_csv"]
    ):
        label_matches += 1

print(
    "Identifiants correspondant à test.csv :",
    f"{id_matches}/335",
)

print(
    "Labels correspondant à test.csv :",
    f"{label_matches}/335",
)

assert label_matches == 335, (
    "L'ordre des prédictions ne correspond pas à test.csv."
)

# ============================================================
# 6. MÉTRIQUES GLOBALES
# ============================================================

y_true = np.array([
    record["true_label"]
    for record in records
])

y_pred = np.array([
    record["predicted_label"]
    for record in records
])

top5_correct = np.array([
    record["top5_correct"]
    for record in records
])

accuracy = accuracy_score(
    y_true,
    y_pred,
)

top5_accuracy = top5_correct.mean()

print("\n" + "=" * 70)
print("RÉSULTATS CORRECTS SUR LES 335 FICHIERS TEST")
print("=" * 70)

print(
    f"Top-1 accuracy : {accuracy * 100:.3f}%"
)

print(
    f"Top-5 accuracy : {top5_accuracy * 100:.3f}%"
)

print(
    "Prédictions correctes :",
    int((y_true == y_pred).sum()),
    "/",
    len(y_true),
)

print(
    "Erreurs :",
    int((y_true != y_pred).sum()),
    "/",
    len(y_true),
)

# ============================================================
# 7. MÉTRIQUES PAR CLASSE
# ============================================================

report_dict = classification_report(
    y_true,
    y_pred,
    labels=labels,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

report_text = classification_report(
    y_true,
    y_pred,
    labels=labels,
    target_names=class_names,
    digits=4,
    zero_division=0,
)

print("\nCLASSIFICATION REPORT\n")
print(report_text)

per_class_rows = []

for class_name in class_names:
    metrics = report_dict[class_name]

    per_class_rows.append({
        "class_name": class_name,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1-score"],
        "support": int(metrics["support"]),
    })

per_class_df = pd.DataFrame(per_class_rows)

per_class_df.to_csv(
    OUTPUT_DIR / "per_class_metrics.csv",
    index=False,
)

# ============================================================
# 8. MATRICE DE CONFUSION
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels,
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names,
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix_counts.csv"
)

# Matrice normalisée par vraie classe
row_sums = cm.sum(
    axis=1,
    keepdims=True,
)

cm_normalized = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(
        cm,
        dtype=float,
    ),
    where=row_sums != 0,
)

cm_normalized_df = pd.DataFrame(
    cm_normalized,
    index=class_names,
    columns=class_names,
)

cm_normalized_df.to_csv(
    OUTPUT_DIR / "confusion_matrix_normalized.csv"
)

# ============================================================
# 9. TABLEAU DE TOUTES LES PRÉDICTIONS
# ============================================================

prediction_rows = []

for index, (record, test_row) in enumerate(
    zip(records, test_rows)
):
    prediction_rows.append({
        "index": index,
        "video_path": test_row["path"],
        "video_id": record["video_id"],
        "true_label": record["true_label"],
        "true_class": class_names[
            record["true_label"]
        ],
        "predicted_label": record["predicted_label"],
        "predicted_class": class_names[
            record["predicted_label"]
        ],
        "correct": record["correct"],
        "top5_correct": record["top5_correct"],
        "confidence": record["confidence"],
    })

predictions_df = pd.DataFrame(
    prediction_rows
)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False,
)

# Fichier contenant uniquement les erreurs
errors_df = predictions_df[
    predictions_df["correct"] == 0
].copy()

errors_df.to_csv(
    OUTPUT_DIR / "test_errors.csv",
    index=False,
)

# ============================================================
# 10. GRAPHIQUE DE LA MATRICE DE CONFUSION
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 11)
)

image = ax.imshow(
    cm_normalized
)

ax.set_xticks(
    range(len(class_names))
)

ax.set_yticks(
    range(len(class_names))
)

ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right",
)

ax.set_yticklabels(
    class_names
)

ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")

ax.set_title(
    "VideoMamba Middle — Matrice de confusion normalisée"
)

for row in range(len(class_names)):
    for column in range(len(class_names)):
        value = cm_normalized[row, column]

        ax.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
        )

fig.colorbar(
    image,
    ax=ax,
)

fig.tight_layout()

figure_path = (
    OUTPUT_DIR
    / "confusion_matrix_normalized.png"
)

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

# ============================================================
# 11. RÉSUMÉ DES FICHIERS CRÉÉS
# ============================================================

print("\nFichiers d'analyse créés :")

for filename in [
    "per_class_metrics.csv",
    "confusion_matrix_counts.csv",
    "confusion_matrix_normalized.csv",
    "test_predictions.csv",
    "test_errors.csv",
    "confusion_matrix_normalized.png",
]:
    path = OUTPUT_DIR / filename

    print(
        filename,
        "->",
        path.exists(),
    )

In [ ]:
from pathlib import Path

VIDEO_SM_DIR = Path(
    "/kaggle/working/VideoMamba/videomamba/video_sm"
)

SCRIPT = VIDEO_SM_DIR / "run_class_finetuning.py"

print("Dépôt VideoMamba présent :", VIDEO_SM_DIR.exists())
print("Script d'évaluation présent :", SCRIPT.exists())
print("Chemin :", VIDEO_SM_DIR)

assert VIDEO_SM_DIR.exists(), (
    "Le dossier VideoMamba n'a pas été créé."
)

assert SCRIPT.exists(), (
    "Le script run_class_finetuning.py est absent."
)

print("DÉPÔT VIDEOMAMBA VALIDÉ")

In [ ]:
from pathlib import Path

MODEL_FILE = Path(
    "/kaggle/working/VideoMamba/"
    "videomamba/video_sm/models/videomamba.py"
)

text = MODEL_FILE.read_text(encoding="utf-8")

old_code = """
hidden_states = checkpoint.checkpoint(self.mixer, hidden_states, inference_params)
""".strip()

new_code = """
hidden_states = checkpoint.checkpoint(
    self.mixer,
    hidden_states,
    inference_params,
    use_reentrant=False
)
""".strip()

if new_code in text:
    print("Correction déjà appliquée.")
elif old_code in text:
    text = text.replace(old_code, new_code, 1)
    MODEL_FILE.write_text(text, encoding="utf-8")
    print("Checkpoint corrigé avec use_reentrant=False.")
else:
    raise RuntimeError(
        "La ligne checkpoint n'a pas été trouvée."
    )

print("CORRECTION CHECKPOINT TERMINÉE")

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import os
import shutil
import subprocess

# --------------------------------------------------
# Chemins
# --------------------------------------------------
PY = "/kaggle/working/vm_env/bin/python"

VIDEO_SM_DIR = Path(
    "/kaggle/working/VideoMamba/videomamba/video_sm"
)

ANNO_DIR = Path("/kaggle/working/annotations")
SMOKE_ANNO_DIR = Path("/kaggle/working/annotations_smoke")
SMOKE_OUTPUT_DIR = Path(
    "/kaggle/working/videomamba_smoke_output"
)

CHECKPOINT_PATH = Path(
    "/kaggle/working/checkpoints/"
    "videomamba_m16_k400_f16_res224.pth"
)

assert VIDEO_SM_DIR.exists()
assert CHECKPOINT_PATH.exists()

# Recrée uniquement les dossiers du petit test
shutil.rmtree(
    SMOKE_ANNO_DIR,
    ignore_errors=True,
)

shutil.rmtree(
    SMOKE_OUTPUT_DIR,
    ignore_errors=True,
)

SMOKE_ANNO_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SMOKE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# --------------------------------------------------
# Lecture des annotations
# --------------------------------------------------
def read_annotations(path):
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as file:
        reader = csv.reader(file)

        for row in reader:
            if len(row) != 2:
                raise ValueError(
                    f"Ligne incorrecte dans "
                    f"{path.name} : {row}"
                )

            video_path = row[0]
            label = int(row[1])

            if not Path(video_path).exists():
                raise FileNotFoundError(
                    video_path
                )

            rows.append(
                (video_path, label)
            )

    return rows


def select_per_class(
    rows,
    number_per_class,
):
    by_class = defaultdict(list)

    for video_path, label in rows:
        by_class[label].append(
            (video_path, label)
        )

    selected = []

    for label in range(12):
        if (
            len(by_class[label])
            < number_per_class
        ):
            raise RuntimeError(
                f"Pas assez de vidéos "
                f"pour la classe {label}"
            )

        selected.extend(
            by_class[label][
                :number_per_class
            ]
        )

    return selected


train_rows = read_annotations(
    ANNO_DIR / "train.csv"
)

val_rows = read_annotations(
    ANNO_DIR / "val.csv"
)

test_rows = read_annotations(
    ANNO_DIR / "test.csv"
)

# Petit test :
# 4 vidéos train et 1 vidéo val/test par classe
smoke_train = select_per_class(
    train_rows,
    4,
)

smoke_val = select_per_class(
    val_rows,
    1,
)

smoke_test = select_per_class(
    test_rows,
    1,
)

# --------------------------------------------------
# Écriture des fichiers du petit test
# --------------------------------------------------
def write_annotations(path, rows):
    with path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.writer(file)
        writer.writerows(rows)


write_annotations(
    SMOKE_ANNO_DIR / "train.csv",
    smoke_train,
)

write_annotations(
    SMOKE_ANNO_DIR / "val.csv",
    smoke_val,
)

write_annotations(
    SMOKE_ANNO_DIR / "test.csv",
    smoke_test,
)

print(
    "Petit train :",
    len(smoke_train),
)

print(
    "Petite validation :",
    len(smoke_val),
)

print(
    "Petit test :",
    len(smoke_test),
)

# --------------------------------------------------
# Commande VideoMamba
# --------------------------------------------------
command = [
    PY,

    # Initialisation distribuée sur un seul GPU
    "-m",
    "torch.distributed.launch",
    "--nproc_per_node=1",
    "--use_env",
    "--master_port=29501",

    "run_class_finetuning.py",

    "--model",
    "videomamba_middle",

    "--finetune",
    str(CHECKPOINT_PATH),

    "--delete_head",

    "--data_path",
    str(SMOKE_ANNO_DIR),

    "--prefix",
    "",

    "--data_set",
    "Kinetics_sparse",

    "--split",
    ",",

    "--nb_classes",
    "12",

    "--output_dir",
    str(SMOKE_OUTPUT_DIR),

    "--log_dir",
    str(SMOKE_OUTPUT_DIR),

    "--batch_size",
    "1",

    "--update_freq",
    "4",

    "--num_sample",
    "1",

    "--input_size",
    "224",

    "--short_side_size",
    "224",

    "--num_frames",
    "16",

    "--orig_t_size",
    "16",

    "--tubelet_size",
    "1",

    "--sampling_rate",
    "4",

    "--epochs",
    "1",

    "--warmup_epochs",
    "0",

    "--lr",
    "1e-3",

    "--min_lr",
    "1e-6",

    "--opt",
    "adamw",

    "--opt_betas",
    "0.9",
    "0.999",

    "--weight_decay",
    "0.05",

    "--drop_path",
    "0.1",

    # Désactivation Mixup/CutMix
    "--mixup",
    "0",

    "--cutmix",
    "0",

    "--smoothing",
    "0.1",

    "--num_workers",
    "2",

    "--test_num_segment",
    "1",

    "--test_num_crop",
    "1",

    "--dist_eval",

    # Réduction mémoire
    "--use_checkpoint",

    "--checkpoint_num",
    "16",

    # Aucun checkpoint pendant le test
    "--no_save_ckpt",

    "--no_auto_resume",
]

environment = os.environ.copy()

# Utilisation d'un seul GPU
environment["CUDA_VISIBLE_DEVICES"] = "0"
environment["OMP_NUM_THREADS"] = "1"

print(
    "\nLANCEMENT DU TEST COURT "
    "VIDEOMAMBA MIDDLE\n"
)

subprocess.run(
    command,
    cwd=str(VIDEO_SM_DIR),
    env=environment,
    check=True,
)

print(
    "\nTEST COURT VIDEOMAMBA "
    "MIDDLE TERMINÉ"
)

In [ ]:
# ============================================================
# CELLULE 18
# Préparer uniquement le meilleur checkpoint VideoMamba
# ============================================================

from pathlib import Path
import shutil
import torch
import gc

INPUT_ROOT = Path("/kaggle/input")

EVAL_ROOT = Path(
    "/kaggle/working/videomamba_final_evaluation"
)
EVAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BEST_DESTINATION = (
    EVAL_ROOT / "videomamba_middle_best.pth"
)

# Chercher précisément checkpoint-best (8).pth
best_candidates = list(
    INPUT_ROOT.rglob("checkpoint-best (8).pth")
)

print("Checkpoints trouvés :")

for path in best_candidates:
    print(
        " -",
        path,
        f"({path.stat().st_size / 1024**2:.2f} MiB)",
    )

assert len(best_candidates) == 1, (
    "Il faut trouver exactement un fichier "
    "'checkpoint-best (8).pth'. "
    f"Nombre trouvé : {len(best_candidates)}"
)

BEST_SOURCE = best_candidates[0]

# Copier dans /kaggle/working
shutil.copy2(
    BEST_SOURCE,
    BEST_DESTINATION,
)

# Vérifier que c’est bien VideoMamba Middle
checkpoint = torch.load(
    BEST_DESTINATION,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(checkpoint, dict)
assert "model" in checkpoint
assert "args" in checkpoint

saved_args = checkpoint["args"]
state_dict = checkpoint["model"]

model_name = getattr(
    saved_args,
    "model",
    None,
)

saved_epoch = checkpoint.get("epoch")

head_shape = tuple(
    state_dict["head.weight"].shape
)

print("\n" + "=" * 70)
print("CHECKPOINT VIDEOMAMBA VÉRIFIÉ")
print("=" * 70)

print("Source             :", BEST_SOURCE)
print("Destination        :", BEST_DESTINATION)
print("Modèle             :", model_name)
print("Epoch sauvegardée  :", saved_epoch)
print("Tête finale        :", head_shape)
print(
    "Taille             :",
    f"{BEST_DESTINATION.stat().st_size / 1024**2:.2f} MiB",
)

assert model_name == "videomamba_middle"
assert saved_epoch == 27
assert head_shape == (12, 576)

del checkpoint
del state_dict
gc.collect()

print("\nCHECKPOINT BEST VIDEOMAMBA PRÊT")

In [ ]:
from pathlib import Path
import os
import subprocess

# ==================================================
# CHEMINS
# ==================================================
PY = "/kaggle/working/vm_env/bin/python"

VIDEO_SM_DIR = Path(
    "/kaggle/working/VideoMamba/videomamba/video_sm"
)

ANNOTATIONS_DIR = Path(
    "/kaggle/working/annotations"
)

CHECKPOINT_PATH = Path(
    "/kaggle/working/checkpoints/"
    "videomamba_m16_k400_f16_res224.pth"
)

OUTPUT_DIR = Path(
    "/kaggle/working/videomamba_middle_40epochs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ==================================================
# VÉRIFICATIONS
# ==================================================
assert VIDEO_SM_DIR.exists(), VIDEO_SM_DIR
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH

for filename in [
    "train.csv",
    "val.csv",
    "test.csv",
]:
    path = ANNOTATIONS_DIR / filename
    assert path.exists(), path

# Vérification de la correction gradient checkpointing
MODEL_FILE = VIDEO_SM_DIR / "models" / "videomamba.py"
model_text = MODEL_FILE.read_text(encoding="utf-8")

assert "use_reentrant=False" in model_text, (
    "La correction use_reentrant=False est absente."
)

# ==================================================
# OPTIMISATION DDP
# ==================================================
# Le test a confirmé qu'aucun paramètre n'est inutilisé.
# On évite donc le parcours inutile du graphe à chaque batch.
RUN_FILE = VIDEO_SM_DIR / "run_class_finetuning.py"
run_text = RUN_FILE.read_text(encoding="utf-8")

if "find_unused_parameters=True" in run_text:
    run_text = run_text.replace(
        "find_unused_parameters=True",
        "find_unused_parameters=False",
        1,
    )

    RUN_FILE.write_text(
        run_text,
        encoding="utf-8",
    )

    print("DDP optimisé : find_unused_parameters=False")
else:
    print("DDP déjà optimisé.")

# ==================================================
# NOMBRE DE GPU
# ==================================================
gpu_count = int(
    subprocess.check_output(
        [
            PY,
            "-c",
            "import torch; print(torch.cuda.device_count())",
        ],
        text=True,
    ).strip()
)

print("GPU détectés :", gpu_count)

assert gpu_count >= 1, "Aucun GPU détecté."

# Utilise les deux Tesla T4 disponibles
number_of_processes = min(2, gpu_count)

# ==================================================
# COMMANDE D'ENTRAÎNEMENT
# ==================================================
command = [
    PY,

    "-m",
    "torch.distributed.launch",

    f"--nproc_per_node={number_of_processes}",
    "--use_env",
    "--master_port=29502",

    "run_class_finetuning.py",

    # --------------------------
    # Modèle
    # --------------------------
    "--model",
    "videomamba_middle",

    "--finetune",
    str(CHECKPOINT_PATH),

    "--delete_head",

    "--nb_classes",
    "12",

    # --------------------------
    # Dataset complet
    # --------------------------
    "--data_path",
    str(ANNOTATIONS_DIR),

    "--prefix",
    "",

    "--data_set",
    "Kinetics_sparse",

    "--split",
    ",",

    # --------------------------
    # Sorties
    # --------------------------
    "--output_dir",
    str(OUTPUT_DIR),

    "--log_dir",
    str(OUTPUT_DIR),

    # --------------------------
    # Batch
    # 4 vidéos par GPU
    # Batch total = 8
    # --------------------------
    "--batch_size",
    "4",

    "--update_freq",
    "1",

    "--num_sample",
    "1",

    # --------------------------
    # Vidéos
    # --------------------------
    "--input_size",
    "224",

    "--short_side_size",
    "224",

    "--num_frames",
    "16",

    "--orig_t_size",
    "16",

    "--tubelet_size",
    "1",

    "--sampling_rate",
    "4",

    # --------------------------
    # Entraînement
    # --------------------------
    "--epochs",
    "40",

    "--warmup_epochs",
    "5",

    "--lr",
    "1e-3",

    "--warmup_lr",
    "1e-6",

    "--min_lr",
    "1e-6",

    "--opt",
    "adamw",

    "--opt_betas",
    "0.9",
    "0.999",

    "--weight_decay",
    "0.05",

    "--layer_decay",
    "0.75",

    "--drop_path",
    "0.1",

    # --------------------------
    # Augmentations
    # --------------------------
    "--mixup",
    "0.8",

    "--cutmix",
    "1.0",

    "--smoothing",
    "0.1",

    "--reprob",
    "0.25",

    # --------------------------
    # DataLoader
    # 2 workers par GPU
    # --------------------------
    "--num_workers",
    "2",

    # --------------------------
    # Validation et test
    # --------------------------
    "--test_num_segment",
    "1",

    "--test_num_crop",
    "1",

    "--dist_eval",

    # --------------------------
    # Réduction mémoire
    # --------------------------
    "--use_checkpoint",

    "--checkpoint_num",
    "16",

    # --------------------------
    # Sauvegarde et reprise
    # --------------------------
    "--save_ckpt",

    "--auto_resume",

    "--test_best",
]

# ==================================================
# ENVIRONNEMENT
# ==================================================
environment = os.environ.copy()

if number_of_processes == 2:
    environment["CUDA_VISIBLE_DEVICES"] = "0,1"
else:
    environment["CUDA_VISIBLE_DEVICES"] = "0"

environment["OMP_NUM_THREADS"] = "1"

print("\n========================================")
print("LANCEMENT DU VRAI ENTRAÎNEMENT")
print("Modèle       : VideoMamba Middle")
print("Époques      : 40")
print("Train        : 2 578 vidéos")
print("Validation   : 335 vidéos")
print("Test         : 335 vidéos")
print("GPU utilisés :", number_of_processes)
print("Dossier      :", OUTPUT_DIR)
print("========================================\n")

# ==================================================
# LANCEMENT
# ==================================================
subprocess.run(
    command,
    cwd=str(VIDEO_SM_DIR),
    env=environment,
    check=True,
)

# ==================================================
# RÉSULTATS
# ==================================================
print("\n========================================")
print("ENTRAÎNEMENT DES 40 ÉPOQUES TERMINÉ")
print("========================================")

for filename in [
    "checkpoint-best.pth",
    "checkpoint-latest.pth",
    "log.txt",
]:
    path = OUTPUT_DIR / filename

    print(
        filename,
        "->",
        "PRÉSENT" if path.exists() else "ABSENT",
    )

In [ ]:
from pathlib import Path

for fichier in Path("/kaggle/input").rglob("*.pth"):
    print(fichier)

## Prédiction

In [ ]:
# ============================================================
# VM1 — PRÉPARER LE CHECKPOINT BEST VIDEOMAMBA
# ============================================================

from pathlib import Path
import shutil
import torch

PY = "/kaggle/working/vm_env/bin/python"

REPO_DIR = Path("/kaggle/working/VideoMamba")
VIDEO_SM_DIR = REPO_DIR / "videomamba" / "video_sm"
RUN_FILE = VIDEO_SM_DIR / "run_class_finetuning.py"

ANNO_DIR = Path("/kaggle/working/annotations")
INPUT_ROOT = Path("/kaggle/input")

EVAL_ROOT = Path("/kaggle/working/videomamba_final_evaluation")
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = (
    EVAL_ROOT / "videomamba_middle_best.pth"
)

# ------------------------------------------------------------
# Vérifier que les anciennes cellules nécessaires ont été lancées
# ------------------------------------------------------------

assert Path(PY).exists(), (
    "L'environnement vm_env n'existe pas. "
    "Relance d'abord les cellules d'installation."
)

assert VIDEO_SM_DIR.exists(), (
    "Le dépôt VideoMamba n'existe pas. "
    "Relance d'abord la cellule de clonage et de correction."
)

assert RUN_FILE.exists(), RUN_FILE

for filename in ["train.csv", "val.csv", "test.csv"]:
    assert (ANNO_DIR / filename).exists(), (
        f"Fichier d'annotation absent : {ANNO_DIR / filename}"
    )

# ------------------------------------------------------------
# Retrouver checkpoint-best (8).pth
# ------------------------------------------------------------

checkpoint_candidates = list(
    INPUT_ROOT.rglob("checkpoint-best (8).pth")
)

print("Checkpoint(s) trouvé(s) :")
for path in checkpoint_candidates:
    print(" -", path)

assert len(checkpoint_candidates) == 1, (
    "Il faut trouver exactement un checkpoint-best (8).pth. "
    f"Nombre trouvé : {len(checkpoint_candidates)}"
)

UPLOADED_BEST = checkpoint_candidates[0]

shutil.copy2(
    UPLOADED_BEST,
    BEST_CHECKPOINT_PATH,
)

# ------------------------------------------------------------
# Vérification du contenu
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(checkpoint, dict)
assert "model" in checkpoint
assert "args" in checkpoint

saved_args = checkpoint["args"]
state_dict = checkpoint["model"]

model_name = getattr(saved_args, "model", None)
saved_epoch = checkpoint.get("epoch")
head_shape = tuple(state_dict["head.weight"].shape)

assert model_name == "videomamba_middle"
assert saved_epoch == 27
assert head_shape == (12, 576)

print("\nModèle            :", model_name)
print("Epoch sauvegardée :", saved_epoch)
print("Tête finale       :", head_shape)
print("Checkpoint        :", BEST_CHECKPOINT_PATH)
print(
    "Taille            :",
    round(BEST_CHECKPOINT_PATH.stat().st_size / 1024**2, 2),
    "MiB",
)

print("\n CHECKPOINT VIDEOMAMBA VALIDÉ")

In [ ]:
# ============================================================
# VM2 — CORRIGER L'ÉVALUATION MONO-GPU
# ============================================================

from pathlib import Path

RUN_FILE = Path(
    "/kaggle/working/VideoMamba/"
    "videomamba/video_sm/run_class_finetuning.py"
)

assert RUN_FILE.exists(), RUN_FILE

run_text = RUN_FILE.read_text(encoding="utf-8")

# ------------------------------------------------------------
# 1. Ajouter sampler_test en mode non distribué
# ------------------------------------------------------------

single_gpu_test_line = (
    "sampler_test = "
    "torch.utils.data.SequentialSampler(dataset_test)"
)

if single_gpu_test_line not in run_text:

    old_sampler_block = """    else:
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
"""

    new_sampler_block = """    else:
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
        sampler_test = torch.utils.data.SequentialSampler(dataset_test)
"""

    assert old_sampler_block in run_text, (
        "Bloc sampler_val introuvable dans run_class_finetuning.py"
    )

    run_text = run_text.replace(
        old_sampler_block,
        new_sampler_block,
        1,
    )

    print("1 sampler_test ajouté pour un seul GPU")

else:
    print("1 sampler_test était déjà corrigé")

# ------------------------------------------------------------
# 2. Sécuriser toutes les barrières distribuées
# ------------------------------------------------------------

unsafe_barrier = "torch.distributed.barrier()"

safe_barrier = (
    "(torch.distributed.barrier() "
    "if torch.distributed.is_available() "
    "and torch.distributed.is_initialized() "
    "else None)"
)

if unsafe_barrier in run_text:
    barrier_count = run_text.count(unsafe_barrier)

    run_text = run_text.replace(
        unsafe_barrier,
        safe_barrier,
    )

    print(
        f"1 {barrier_count} barrière(s) distribuée(s) sécurisée(s)"
    )
else:
    print("1 Les barrières étaient déjà sécurisées")

RUN_FILE.write_text(
    run_text,
    encoding="utf-8",
)

# ------------------------------------------------------------
# Vérifications
# ------------------------------------------------------------

final_text = RUN_FILE.read_text(encoding="utf-8")

assert single_gpu_test_line in final_text
assert unsafe_barrier not in final_text

print("\n1 SCRIPT D'ÉVALUATION MONO-GPU PRÊT")

In [ ]:
# ============================================================
# VM2 BIS — VÉRIFIER LA CORRECTION MONO-GPU
# ============================================================

from pathlib import Path

RUN_FILE = Path(
    "/kaggle/working/VideoMamba/"
    "videomamba/video_sm/run_class_finetuning.py"
)

assert RUN_FILE.exists(), RUN_FILE

final_text = RUN_FILE.read_text(
    encoding="utf-8",
)

single_gpu_test_line = (
    "sampler_test = "
    "torch.utils.data.SequentialSampler(dataset_test)"
)

assert single_gpu_test_line in final_text, (
    "sampler_test mono-GPU est absent."
)

barrier_lines = [
    line.strip()
    for line in final_text.splitlines()
    if "torch.distributed.barrier()" in line
]

print("Lignes contenant barrier :")

for line in barrier_lines:
    print(" -", line)

# Une barrière non protégée serait une ligne contenant uniquement :
# torch.distributed.barrier()
unsafe_standalone_barriers = [
    line
    for line in barrier_lines
    if line == "torch.distributed.barrier()"
]

assert len(unsafe_standalone_barriers) == 0, (
    "Il reste une barrière distribuée non protégée."
)

assert len(barrier_lines) >= 1
assert all(
    "torch.distributed.is_initialized()" in line
    for line in barrier_lines
)

print("\nSCRIPT D'ÉVALUATION MONO-GPU PRÊT")

In [ ]:
# ============================================================
# VM3 — PRÉPARER LES DEUX JEUX D'ÉVALUATION
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

SOURCE_ANNO_DIR = Path("/kaggle/working/annotations")

VAL_EVAL_ANNO_DIR = Path(
    "/kaggle/working/videomamba_annotations_validation"
)

TEST_EVAL_ANNO_DIR = Path(
    "/kaggle/working/videomamba_annotations_test"
)

VAL_EVAL_ANNO_DIR.mkdir(parents=True, exist_ok=True)
TEST_EVAL_ANNO_DIR.mkdir(parents=True, exist_ok=True)


def count_rows(path: Path) -> int:
    dataframe = pd.read_csv(
        path,
        header=None,
    )
    return len(dataframe)


def prepare_evaluation_annotations(
    destination_dir: Path,
    target_csv: Path,
):
    """
    Le script construit train, val et test même en mode --eval.
    On fournit donc les trois fichiers.

    val.csv et test.csv contiennent ici le split à évaluer.
    """

    shutil.copy2(
        SOURCE_ANNO_DIR / "train.csv",
        destination_dir / "train.csv",
    )

    shutil.copy2(
        target_csv,
        destination_dir / "val.csv",
    )

    shutil.copy2(
        target_csv,
        destination_dir / "test.csv",
    )


prepare_evaluation_annotations(
    destination_dir=VAL_EVAL_ANNO_DIR,
    target_csv=SOURCE_ANNO_DIR / "val.csv",
)

prepare_evaluation_annotations(
    destination_dir=TEST_EVAL_ANNO_DIR,
    target_csv=SOURCE_ANNO_DIR / "test.csv",
)

assert count_rows(
    VAL_EVAL_ANNO_DIR / "test.csv"
) == 335

assert count_rows(
    TEST_EVAL_ANNO_DIR / "test.csv"
) == 335

print("Validation à évaluer :", count_rows(
    VAL_EVAL_ANNO_DIR / "test.csv"
))

print("Test à évaluer       :", count_rows(
    TEST_EVAL_ANNO_DIR / "test.csv"
))

print("\nDossier validation :", VAL_EVAL_ANNO_DIR)
print("Dossier test       :", TEST_EVAL_ANNO_DIR)

print("\n1 ANNOTATIONS D'ÉVALUATION PRÊTES")

In [ ]:
# ============================================================
# VM4 — CONSTRUIRE LES COMMANDES D'ÉVALUATION
# ============================================================

from pathlib import Path
import os

PY = "/kaggle/working/vm_env/bin/python"

VIDEO_SM_DIR = Path(
    "/kaggle/working/VideoMamba/videomamba/video_sm"
)

RUN_FILE = VIDEO_SM_DIR / "run_class_finetuning.py"

EVAL_ROOT = Path(
    "/kaggle/working/videomamba_final_evaluation"
)

BEST_CHECKPOINT_PATH = (
    EVAL_ROOT / "videomamba_middle_best.pth"
)

VAL_OUTPUT_DIR = EVAL_ROOT / "validation"
TEST_OUTPUT_DIR = EVAL_ROOT / "test"

VAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def build_videomamba_eval_command(
    annotations_dir: Path,
    output_dir: Path,
):
    return [
        PY,
        str(RUN_FILE),

        # Évaluation uniquement
        "--eval",
        "--no_auto_resume",
        "--no_save_ckpt",

        # Charger directement notre checkpoint entraîné
        "--finetune",
        str(BEST_CHECKPOINT_PATH),

        "--model_key",
        "model",

        # Architecture
        "--model",
        "videomamba_middle",

        "--nb_classes",
        "12",

        # Données
        "--data_path",
        str(annotations_dir),

        "--prefix",
        "",

        "--data_set",
        "Kinetics_sparse",

        "--split",
        ",",

        # Sorties
        "--output_dir",
        str(output_dir),

        "--log_dir",
        str(output_dir),

        # Inférence
        "--batch_size",
        "2",

        "--num_sample",
        "1",

        "--num_workers",
        "2",

        "--input_size",
        "224",

        "--short_side_size",
        "224",

        "--num_frames",
        "16",

        "--orig_t_size",
        "16",

        "--tubelet_size",
        "1",

        "--sampling_rate",
        "4",

        "--test_num_segment",
        "1",

        "--test_num_crop",
        "1",

        "--drop_path",
        "0.1",

        "--device",
        "cuda",
    ]


VAL_COMMAND = build_videomamba_eval_command(
    annotations_dir=VAL_EVAL_ANNO_DIR,
    output_dir=VAL_OUTPUT_DIR,
)

TEST_COMMAND = build_videomamba_eval_command(
    annotations_dir=TEST_EVAL_ANNO_DIR,
    output_dir=TEST_OUTPUT_DIR,
)

EVAL_ENV = os.environ.copy()
EVAL_ENV["CUDA_VISIBLE_DEVICES"] = "0"
EVAL_ENV["OMP_NUM_THREADS"] = "1"

print("Commande validation prête.")
print("Commande test prête.")

print("\nGPU prévu : un seul GPU, CUDA_VISIBLE_DEVICES=0")
print("\n1 COMMANDES D'ÉVALUATION PRÊTES")

In [ ]:
# ============================================================
# VM5 — LANCER VALIDATION PUIS TEST
# ============================================================

from pathlib import Path
import shutil
import subprocess


def run_and_save_console(
    command,
    working_directory: Path,
    environment,
    log_path: Path,
):
    """
    Affiche la sortie en direct et la sauvegarde dans un fichier.
    """

    print("\n" + "=" * 90)
    print("LANCEMENT :", log_path.stem)
    print("=" * 90 + "\n")

    process = subprocess.Popen(
        command,
        cwd=str(working_directory),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_file:

        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"L'évaluation a échoué. Code retour : {return_code}. "
            f"Consulte : {log_path}"
        )

    print("\n1 Évaluation terminée :", log_path.stem)


# ------------------------------------------------------------
# Nettoyer seulement les anciens résultats d'évaluation
# ------------------------------------------------------------

for output_directory in [
    VAL_OUTPUT_DIR,
    TEST_OUTPUT_DIR,
]:
    if output_directory.exists():
        shutil.rmtree(output_directory)

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

run_and_save_console(
    command=VAL_COMMAND,
    working_directory=VIDEO_SM_DIR,
    environment=EVAL_ENV,
    log_path=EVAL_ROOT / "validation_console.log",
)

# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

run_and_save_console(
    command=TEST_COMMAND,
    working_directory=VIDEO_SM_DIR,
    environment=EVAL_ENV,
    log_path=EVAL_ROOT / "test_console.log",
)

# ------------------------------------------------------------
# Vérifier les fichiers bruts
# ------------------------------------------------------------

VAL_RAW_FILE = VAL_OUTPUT_DIR / "0.txt"
TEST_RAW_FILE = TEST_OUTPUT_DIR / "0.txt"

assert VAL_RAW_FILE.exists(), VAL_RAW_FILE
assert TEST_RAW_FILE.exists(), TEST_RAW_FILE

print("\nFichier brut validation :", VAL_RAW_FILE)
print("Fichier brut test       :", TEST_RAW_FILE)

print("\n1 LES DEUX INFÉRENCES SONT TERMINÉES")

In [ ]:
# ============================================================
# VM6 — CONVERTIR LES SORTIES BRUTES EN CSV
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

CLASS_NAMES = [
    "Jumping",
    "child_running",
    "climbing",
    "collesion",
    "electrical_appliances",
    "falling",
    "head_stuck",
    "normal",
    "objects_falling",
    "pulling_curtains",
    "scribbling",
    "sharp_objects",
]

NUM_CLASSES = len(CLASS_NAMES)

assert NUM_CLASSES == 12


def stable_softmax(logits):
    logits = np.asarray(
        logits,
        dtype=np.float64,
    )

    shifted = logits - np.max(logits)
    exponentials = np.exp(shifted)

    return exponentials / exponentials.sum()


def parse_raw_prediction_line(line: str):
    """
    Format officiel :
    video_id [logit_0, ..., logit_11] label chunk split
    """

    prefix, remaining = line.split("[", 1)
    logits_text, suffix = remaining.split("]", 1)

    raw_video_id = prefix.strip()

    logits = np.fromstring(
        logits_text,
        dtype=np.float64,
        sep=",",
    )

    suffix_parts = suffix.strip().split()

    assert len(suffix_parts) >= 3, line

    true_label = int(suffix_parts[0])
    chunk_number = int(suffix_parts[1])
    split_number = int(suffix_parts[2])

    assert len(logits) == NUM_CLASSES, (
        f"{len(logits)} logits trouvés au lieu de {NUM_CLASSES}"
    )

    return {
        "raw_video_id": raw_video_id,
        "logits": logits,
        "true_label": true_label,
        "chunk_number": chunk_number,
        "split_number": split_number,
    }


def export_predictions(
    raw_file: Path,
    annotation_file: Path,
    split_name: str,
):
    # Le premier élément du fichier brut contient les scores du dernier batch.
    raw_lines = raw_file.read_text(
        encoding="utf-8"
    ).splitlines()[1:]

    annotations = pd.read_csv(
        annotation_file,
        header=None,
        names=[
            "video_path",
            "annotation_label",
        ],
    )

    print(f"\n{split_name.upper()}")
    print("Lignes brutes :", len(raw_lines))
    print("Annotations   :", len(annotations))

    assert len(raw_lines) == 335, (
        f"{split_name} : {len(raw_lines)} lignes brutes au lieu de 335. "
        "Vérifie test_num_segment=1, test_num_crop=1 et le mono-GPU."
    )

    assert len(annotations) == 335

    rows = []

    for row_index, raw_line in enumerate(raw_lines):
        parsed = parse_raw_prediction_line(raw_line)

        annotation_row = annotations.iloc[row_index]

        annotation_label = int(
            annotation_row["annotation_label"]
        )

        # Vérification essentielle de l'alignement
        assert parsed["true_label"] == annotation_label, (
            f"Désalignement à la ligne {row_index}: "
            f"sortie={parsed['true_label']}, "
            f"annotation={annotation_label}"
        )

        probabilities = stable_softmax(
            parsed["logits"]
        )

        predicted_label = int(
            probabilities.argmax()
        )

        confidence = float(
            probabilities[predicted_label]
        )

        top5_labels = np.argsort(
            -probabilities
        )[:5]

        output_row = {
            "video_path": str(
                annotation_row["video_path"]
            ),
            "raw_video_id": parsed["raw_video_id"],
            "true_label": annotation_label,
            "true_class": CLASS_NAMES[annotation_label],
            "predicted_label": predicted_label,
            "predicted_class": CLASS_NAMES[predicted_label],
            "confidence": confidence,
            "correct_top1": int(
                predicted_label == annotation_label
            ),
            "correct_top5": int(
                annotation_label in top5_labels
            ),
        }

        for class_index in range(NUM_CLASSES):
            output_row[f"prob_{class_index}"] = float(
                probabilities[class_index]
            )

        rows.append(output_row)

    predictions_df = pd.DataFrame(rows)

    assert len(predictions_df) == 335
    assert predictions_df["video_path"].nunique() == 335
    assert predictions_df["confidence"].between(0, 1).all()

    output_csv = (
        EVAL_ROOT
        / f"videomamba_{split_name}_predictions.csv"
    )

    predictions_df.to_csv(
        output_csv,
        index=False,
    )

    # --------------------------------------------------------
    # Métriques
    # --------------------------------------------------------

    true_labels = predictions_df[
        "true_label"
    ].to_numpy()

    predicted_labels = predictions_df[
        "predicted_label"
    ].to_numpy()

    top1_accuracy = accuracy_score(
        true_labels,
        predicted_labels,
    )

    top5_accuracy = predictions_df[
        "correct_top5"
    ].mean()

    macro_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="weighted",
        zero_division=0,
    )

    report_dict = classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    report_text = classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    matrix = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
    )

    metrics = {
        "model": "VideoMamba Middle",
        "split": split_name,
        "number_of_videos": int(len(predictions_df)),
        "top1_accuracy": float(top1_accuracy),
        "top5_accuracy": float(top5_accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
    }

    metrics_path = (
        EVAL_ROOT
        / f"videomamba_{split_name}_metrics.json"
    )

    metrics_path.write_text(
        json.dumps(
            metrics,
            indent=2,
        ),
        encoding="utf-8",
    )

    report_json_path = (
        EVAL_ROOT
        / f"videomamba_{split_name}_classification_report.json"
    )

    report_json_path.write_text(
        json.dumps(
            report_dict,
            indent=2,
        ),
        encoding="utf-8",
    )

    report_text_path = (
        EVAL_ROOT
        / f"videomamba_{split_name}_classification_report.txt"
    )

    report_text_path.write_text(
        report_text,
        encoding="utf-8",
    )

    matrix_path = (
        EVAL_ROOT
        / f"videomamba_{split_name}_confusion_matrix.csv"
    )

    pd.DataFrame(
        matrix,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    ).to_csv(matrix_path)

    print("\nRésultats :")
    print(f"Top-1      : {top1_accuracy * 100:.3f}%")
    print(f"Top-5      : {top5_accuracy * 100:.3f}%")
    print(f"Macro F1   : {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")

    print("\nCSV créé :", output_csv)

    return predictions_df, metrics, matrix


# ------------------------------------------------------------
# Export validation
# ------------------------------------------------------------

videomamba_validation_df, videomamba_validation_metrics, val_matrix = (
    export_predictions(
        raw_file=VAL_OUTPUT_DIR / "0.txt",
        annotation_file=VAL_EVAL_ANNO_DIR / "test.csv",
        split_name="validation",
    )
)

# ------------------------------------------------------------
# Export test
# ------------------------------------------------------------

videomamba_test_df, videomamba_test_metrics, test_matrix = (
    export_predictions(
        raw_file=TEST_OUTPUT_DIR / "0.txt",
        annotation_file=TEST_EVAL_ANNO_DIR / "test.csv",
        split_name="test",
    )
)

print("\n1 CSV ET MÉTRIQUES VIDEOMAMBA CRÉÉS")

In [ ]:
# ============================================================
# VM7 — GRAPHIQUES ET TABLEAU RÉCAPITULATIF
# ============================================================

import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay


def save_confusion_matrix(
    matrix,
    split_name,
):
    figure, axis = plt.subplots(
        figsize=(12, 10)
    )

    display_object = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=CLASS_NAMES,
    )

    display_object.plot(
        ax=axis,
        xticks_rotation=45,
        values_format="d",
    )

    axis.set_title(
        f"VideoMamba Middle — {split_name}"
    )

    figure.tight_layout()

    output_path = (
        EVAL_ROOT
        / f"videomamba_{split_name}_confusion_matrix.png"
    )

    figure.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()

    print("Graphique sauvegardé :", output_path)


save_confusion_matrix(
    val_matrix,
    "validation",
)

save_confusion_matrix(
    test_matrix,
    "test",
)

summary_df = pd.DataFrame(
    [
        videomamba_validation_metrics,
        videomamba_test_metrics,
    ]
)

summary_path = (
    EVAL_ROOT
    / "videomamba_validation_test_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

display(summary_df)

print("\nRésumé sauvegardé :", summary_path)
print("\n1 ÉVALUATION VIDEOMAMBA TERMINÉE")

In [ ]:
# ============================================================
# VM8 — CRÉER L'ARCHIVE DES RÉSULTATS
# ============================================================

from pathlib import Path
import shutil

EVAL_ROOT = Path(
    "/kaggle/working/videomamba_final_evaluation"
)

ARCHIVE_BASE = Path(
    "/kaggle/working/videomamba_final_results"
)

archive_path = shutil.make_archive(
    base_name=str(ARCHIVE_BASE),
    format="zip",
    root_dir=str(EVAL_ROOT),
)

print("Archive créée :", archive_path)
print(
    "Taille :",
    round(Path(archive_path).stat().st_size / 1024**2, 2),
    "MiB",
)

print("\n1 TÉLÉCHARGE videomamba_final_results.zip")

In [ ]:
# ============================================================
# SAUVEGARDE FINALE VIDEOMAMBA DANS UN DATASET KAGGLE
# ============================================================

from pathlib import Path
import json
import os
import shutil
import kagglehub

EVAL_DIR = Path(
    "/kaggle/working/videomamba_final_evaluation"
)

BEST_CHECKPOINT = (
    EVAL_DIR / "videomamba_middle_best.pth"
)

VALIDATION_CSV = (
    EVAL_DIR / "videomamba_validation_predictions.csv"
)

TEST_CSV = (
    EVAL_DIR / "videomamba_test_predictions.csv"
)

RESULTS_ZIP = Path(
    "/kaggle/working/videomamba_final_results.zip"
)

UPLOAD_DIR = Path(
    "/kaggle/working/videomamba_final_dataset_upload"
)

DATASET_HANDLE = (
    "nisrineamalkhnifass/"
    "videomamba-final-rhad-results"
)

# ============================================================
# 1. VÉRIFICATIONS
# ============================================================

for required_file in [
    BEST_CHECKPOINT,
    VALIDATION_CSV,
    TEST_CSV,
]:
    assert required_file.exists(), (
        f"Fichier indispensable absent : {required_file}"
    )

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# 2. AJOUTER LE GROS CHECKPOINT SANS LE RECOPIER
# ============================================================

checkpoint_destination = (
    UPLOAD_DIR / "videomamba_middle_best.pth"
)

try:
    os.link(
        BEST_CHECKPOINT,
        checkpoint_destination,
    )
    print(
        "Lien physique créé :",
        checkpoint_destination.name,
    )
except OSError:
    shutil.copy2(
        BEST_CHECKPOINT,
        checkpoint_destination,
    )
    print(
        "Checkpoint copié :",
        checkpoint_destination.name,
    )

# ============================================================
# 3. COPIER TOUS LES RÉSULTATS UTILES
# ============================================================

patterns_to_copy = [
    "videomamba_*_predictions.csv",
    "videomamba_*_metrics.json",
    "videomamba_*_classification_report.json",
    "videomamba_*_classification_report.txt",
    "videomamba_*_confusion_matrix.csv",
    "videomamba_*_confusion_matrix.png",
    "videomamba_validation_test_summary.csv",
    "validation_console.log",
    "test_console.log",
]

copied_files = set()

for pattern in patterns_to_copy:
    for source_file in EVAL_DIR.glob(pattern):

        if not source_file.is_file():
            continue

        destination_file = (
            UPLOAD_DIR / source_file.name
        )

        # Ne pas recopier le checkpoint déjà ajouté.
        if destination_file.name == (
            checkpoint_destination.name
        ):
            continue

        shutil.copy2(
            source_file,
            destination_file,
        )

        copied_files.add(
            destination_file.name
        )

        print(
            "Résultat ajouté :",
            destination_file.name,
        )

# ============================================================
# 4. AJOUTER L'ARCHIVE SI ELLE EXISTE
# ============================================================

if RESULTS_ZIP.exists():
    shutil.copy2(
        RESULTS_ZIP,
        UPLOAD_DIR / RESULTS_ZIP.name,
    )

    print(
        "Archive ajoutée :",
        RESULTS_ZIP.name,
    )
else:
    print(
        "Archive absente : ce n'est pas bloquant."
    )

# ============================================================
# 5. CRÉER UN RÉSUMÉ CLAIR
# ============================================================

summary = {
    "model": "VideoMamba Middle",
    "checkpoint": "videomamba_middle_best.pth",
    "checkpoint_training_epoch": 27,
    "number_of_classes": 12,
    "number_of_frames": 16,
    "normal_class_index": 7,
    "validation": {
        "number_of_videos": 335,
        "top1_accuracy": 0.850746,
        "top5_accuracy": 0.970149,
        "macro_f1": 0.779226,
        "weighted_f1": 0.842771,
    },
    "test": {
        "number_of_videos": 335,
        "top1_accuracy": 0.862687,
        "top5_accuracy": 0.964179,
        "macro_f1": 0.795159,
        "weighted_f1": 0.857615,
    },
    "files_for_comparison_and_AT": [
        "videomamba_validation_predictions.csv",
        "videomamba_test_predictions.csv",
    ],
}

summary_path = (
    UPLOAD_DIR
    / "videomamba_final_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

# ============================================================
# 6. AFFICHER CE QUI VA ÊTRE ENVOYÉ
# ============================================================

print("\n" + "=" * 75)
print("FICHIERS PRÉPARÉS POUR LE DATASET")
print("=" * 75)

for file_path in sorted(
    UPLOAD_DIR.iterdir()
):
    print(
        " -",
        file_path.name,
        f"({file_path.stat().st_size / 1024**2:.2f} MiB)",
    )

# Vérification finale des fichiers AT.
assert (
    UPLOAD_DIR
    / "videomamba_validation_predictions.csv"
).exists()

assert (
    UPLOAD_DIR
    / "videomamba_test_predictions.csv"
).exists()

assert checkpoint_destination.exists()

# ============================================================
# 7. UPLOAD VERS KAGGLE
# ============================================================

print("\n" + "=" * 75)
print("UPLOAD VIDEOMAMBA FINAL VERS KAGGLE")
print("Ne ferme pas la session avant la confirmation.")
print("=" * 75 + "\n")

kagglehub.dataset_upload(
    DATASET_HANDLE,
    str(UPLOAD_DIR),
    version_notes=(
        "VideoMamba Middle RHAD final: best checkpoint, "
        "validation/test predictions, metrics, confusion "
        "matrices, reports and logs."
    ),
)

print("\n" + "=" * 75)
print("SAUVEGARDE VIDEOMAMBA TERMINÉE")
print("=" * 75)

print("Dataset :", DATASET_HANDLE)

print(
    "Page :",
    "https://www.kaggle.com/datasets/"
    + DATASET_HANDLE,
)

print(
    "\n VIDEOMAMBA EST SÉCURISÉ "
    "ET PRÊT POUR COMPARISON_AND_AT"
)